In [8]:
!pip install pubchempy scikit-learn tqdm pandas

In [9]:
import pandas as pd
import pubchempy as pcp
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter

In [31]:
df = pd.read_csv("./dataset/bio-decagon-combo.csv")  
# columns: Drug1_CID, Drug2_CID, SideEffect

print("Initial rows:", len(df))
df.head()

Initial rows: 4649441


,STITCH 1,STITCH 2,Polypharmacy Side Effect,Side Effect Name
0,CID000002173,CID000003345,C0151714,hypermagnesemia
1,CID000002173,CID000003345,C0035344,retinopathy of prematurity
2,CID000002173,CID000003345,C0004144,atelectasis
3,CID000002173,CID000003345,C0002063,alkalosis
4,CID000002173,CID000003345,C0004604,Back Ache


In [32]:
import pandas as pd
print(df["STITCH 1"].head(5))
print(df["STITCH 2"].head(5))
# Extract only digits from STITCH columns
df["STITCH 1_clean"] = df["STITCH 1"].astype(str).str.extract(r'(\d+)').astype(int)
df["STITCH 2_clean"] = df["STITCH 2"].astype(str).str.extract(r'(\d+)').astype(int)

# Now get unique CIDs
unique_cids = set(df["STITCH 1_clean"]).union(set(df["STITCH 2_clean"]))

print("Unique CIDs:", len(unique_cids))

0    CID000002173
1    CID000002173
2    CID000002173
3    CID000002173
4    CID000002173
Name: STITCH 1, dtype: object
0    CID000003345
1    CID000003345
2    CID000003345
3    CID000003345
4    CID000003345
Name: STITCH 2, dtype: object
Unique CIDs: 645


In [33]:
cid_to_smiles = {}

for cid in tqdm(unique_cids):
    try:
        compound = pcp.Compound.from_cid(cid)
        cid_to_smiles[cid] = compound.canonical_smiles
    except:
        cid_to_smiles[cid] = None

  0%|          | 0/645 [00:00<?, ?it/s]C:\Users\DELL\AppData\Local\Temp\ipykernel_8944\1601910491.py:6: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  cid_to_smiles[cid] = compound.canonical_smiles
100%|██████████| 645/645 [1:20:55<00:00,  7.53s/it]     


In [35]:
mapping_df = pd.DataFrame(
    list(cid_to_smiles.items()),
    columns=["CID", "SMILES"]
)

mapping_df.to_csv("./dataset/cid_smiles_map.csv", index=False)
print("Saved mapping!")

Saved mapping!


In [37]:
df["Drug1_SMILES"] = df["STITCH 1_clean"].map(cid_to_smiles)
df["Drug2_SMILES"] = df["STITCH 2_clean"].map(cid_to_smiles)
df = df.dropna(subset=["Drug1_SMILES", "Drug2_SMILES"])

print("After SMILES:", len(df))

preprocessed_df = df[["Drug1_SMILES", "Drug2_SMILES", "Side Effect Name"]]

# Save to CSV
preprocessed_df.to_csv("./dataset/preprocessed_dataset.csv", index=False)

print("Saved preprocessed dataset!")
preprocessed_df.head()

After SMILES: 4633308
Saved preprocessed dataset!


,Drug1_SMILES,Drug2_SMILES,Side Effect Name
0,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...,CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3,hypermagnesemia
1,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...,CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3,retinopathy of prematurity
2,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...,CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3,atelectasis
3,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...,CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3,alkalosis
4,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=CC=C3)N)C(=O...,CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3,Back Ache


In [38]:
import pandas as pd
import pubchempy as pcp
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter
INPUT_FILE = "./dataset/preprocessed_dataset.csv"
df = pd.read_csv(INPUT_FILE)


In [41]:
# Normalize pairs WITHOUT apply (vectorized)
df["SMILES1"] = df[["Drug1_SMILES", "Drug2_SMILES"]].min(axis=1)
df["SMILES2"] = df[["Drug1_SMILES", "Drug2_SMILES"]].max(axis=1)

# Remove duplicates
df = df.drop_duplicates(subset=["SMILES1", "SMILES2", "Side Effect Name"])

df.to_csv("./dataset/processed_step2.csv", index=False)

print("After dedup:", len(df))

After dedup: 4633308


In [43]:
grouped = (
    df.groupby(["SMILES1", "SMILES2"])["Side Effect Name"]
    .agg(lambda x: list(pd.unique(x)))
    .reset_index()
)
grouped.to_csv("./dataset/processed_step3.csv", index=False)
print("After grouping:", len(grouped))

After grouping: 63305


In [45]:
from collections import Counter
import pandas as pd

# Flatten efficiently
all_effects = grouped["Side Effect Name"].explode()

# Count frequencies
effect_counts = all_effects.value_counts()

# Select frequent ones
selected_effects = set(effect_counts[effect_counts >= 500].index)

# Filter
grouped["Side Effect Name"] = grouped["Side Effect Name"].apply(
    lambda x: [e for e in x if e in selected_effects]
)

# Remove empty rows
grouped = grouped[grouped["Side Effect Name"].map(len) > 0]

# Save
grouped.to_csv("./dataset/final_processed_data.csv", index=False)

print("After filtering:", len(grouped))
print("Number of labels:", len(selected_effects))

After filtering: 63304
Number of labels: 963
